# Fréchet Inception Distance (FID) from Features — Solution

**Difficulty**: 🟢 Easy

---

### Problem Statement

**FID** (Heusel et al., 2017) is the default metric for image generators. Embed $N$ real and $N$ generated images with an Inception-v3 network (2048-d pool features), fit a Gaussian to each feature set, and report the **squared Fréchet (2-Wasserstein) distance between the two Gaussians**:

$$\mathrm{FID} = \lVert \mu_1 - \mu_2 \rVert_2^2 + \operatorname{Tr}\!\Big(\Sigma_1 + \Sigma_2 - 2\,\big(\Sigma_1 \Sigma_2\big)^{1/2}\Big)$$

Lower is better; 0 means the two Gaussians coincide. The network is the easy part — the subtle part is the matrix square root: $\Sigma_1\Sigma_2$ is **not symmetric**, the covariances are often singular ($N <$ feature dim in quick evaluations), and the answer is a small difference of large traces, so naive implementations return complex numbers, NaNs or garbage in float32. Here you skip the network and implement the statistics and the distance on given feature vectors.

---

### Requirements

1. **`feature_statistics(feats)`** — `feats`: `(N, D)` (rows are samples) → `(mu, sigma)` with `mu`: `(D,)` and `sigma`: `(D, D)` the **unbiased** covariance (divide by $N - 1$), i.e. the same as `torch.cov(feats.T)` / `numpy.cov(feats, rowvar=False)`. Must work when $N < D$.
2. **`frechet_distance(mu1, sigma1, mu2, sigma2)`** → a Python `float` (a 0-dim tensor is also accepted):
   - convert all inputs to **float64** first, whatever their dtype;
   - compute $\operatorname{Tr}\big((\Sigma_1\Sigma_2)^{1/2}\big) = \sum_i \sqrt{\lambda_i}$, where $\lambda_i$ are the eigenvalues of the symmetric PSD matrix $M = \Sigma_1^{1/2}\,\Sigma_2\,\Sigma_1^{1/2}$, with $\Sigma_1^{1/2} = V \operatorname{diag}(\sqrt{w})V^\top$ from `torch.linalg.eigh(sigma1)`;
   - clamp negative eigenvalues (round-off) to 0 before every square root;
   - return $\lVert\mu_1-\mu_2\rVert^2 + \operatorname{Tr}\Sigma_1 + \operatorname{Tr}\Sigma_2 - 2\sum_i\sqrt{\lambda_i}$.
   The result must be symmetric in its arguments, and finite and non-negative (up to ~1e-6 round-off) even for singular covariances.

---

### Constraints

- ✅ `torch.linalg.eigh` / `torch.linalg.eigvalsh` on symmetric matrices.
- ❌ No `scipy.linalg.sqrtm`, `torchmetrics`, `pytorch-fid` or `clean-fid`.
- ❌ Don't call `eigh` / `eigvalsh` on $\Sigma_1\Sigma_2$ itself — it is not symmetric and those routines read only one triangle.
- ❌ Don't take an elementwise `sqrt` of a matrix and call it a matrix square root.

---

<details>
  <summary>💡 Hint</summary>

  - Two matrices related by $B = P A P^{-1}$ have the same eigenvalues. Write $\Sigma_1\Sigma_2 = \Sigma_1^{1/2}\big(\Sigma_1^{1/2}\Sigma_2\Sigma_1^{1/2}\big)\Sigma_1^{-1/2}$.
  - The square root of a matrix with eigenvalues $\lambda_i$ has eigenvalues $\sqrt{\lambda_i}$, and the trace is the sum of eigenvalues — you never need the matrix square root itself, only its trace.
  - Symmetrize $M \leftarrow (M + M^\top)/2$ before `eigvalsh`: products of symmetric matrices are only symmetric up to round-off.

</details>

---

In [ ]:
import math

import torch

torch.manual_seed(0)

In [ ]:
# Synthetic "Inception features": correlated Gaussians in D = 64 (no network, no downloads)
D = 64
_A = torch.randn(D, D, generator=torch.Generator().manual_seed(0)) / math.sqrt(D)   # mixing matrix


def real_features(n, seed):
    g = torch.Generator().manual_seed(seed)
    return torch.randn(n, D, generator=g) @ _A.T + 1.0


def fake_features(n, seed, shift=0.0, scale=1.0):
    """A 'generator' whose features are shifted and/or over-dispersed versions of the real ones."""
    g = torch.Generator().manual_seed(seed)
    return scale * (torch.randn(n, D, generator=g) @ _A.T) + 1.0 + shift


real = real_features(5000, seed=1)
print(f"real features {tuple(real.shape)}")

In [ ]:
def feature_statistics(feats):
    mu = feats.mean(dim=0)                            # (D,)
    xc = feats - mu                                   # centre every feature
    sigma = xc.T @ xc / (feats.shape[0] - 1)          # (D, D), unbiased like torch.cov
    return mu, sigma


def _sqrtm_psd(a):
    """Symmetric PSD square root via eigh; round-off negative eigenvalues are clamped to 0."""
    w, v = torch.linalg.eigh(a)
    return (v * w.clamp(min=0).sqrt()) @ v.T          # V diag(sqrt(w)) V^T


def frechet_distance(mu1, sigma1, mu2, sigma2):
    mu1, sigma1, mu2, sigma2 = (torch.as_tensor(t).to(torch.float64) for t in (mu1, sigma1, mu2, sigma2))
    diff = mu1 - mu2
    s1_half = _sqrtm_psd(sigma1)
    m = s1_half @ sigma2 @ s1_half                    # symmetric PSD, same eigenvalues as sigma1 @ sigma2
    m = (m + m.T) / 2                                 # remove round-off asymmetry before eigvalsh
    tr_covmean = torch.linalg.eigvalsh(m).clamp(min=0).sqrt().sum()
    fid = diff @ diff + torch.trace(sigma1) + torch.trace(sigma2) - 2 * tr_covmean
    return float(fid)

In [ ]:
# Test 1: statistics match torch.cov (unbiased), also with fewer samples than dimensions
mu, sigma = feature_statistics(real)
assert mu.shape == (D,) and sigma.shape == (D, D)
assert torch.allclose(mu, real.mean(0), atol=1e-5)
assert torch.allclose(sigma, torch.cov(real.T), atol=1e-4), "use the unbiased (N - 1) covariance"
few = real[:10]
assert torch.allclose(feature_statistics(few)[1], torch.cov(few.T), atol=1e-5)
print("Test 1 passed: feature_statistics")

In [ ]:
# Test 2: closed forms
mu64, s64 = mu.double(), sigma.double()
assert abs(frechet_distance(mu64, s64, mu64, s64)) < 1e-6, "identical statistics -> 0"

delta = torch.linspace(-1, 1, D, dtype=torch.float64)
d = frechet_distance(mu64, s64, mu64 + delta, s64)
assert abs(d - (delta @ delta).item()) < 1e-6, "a pure translation costs ||delta||^2"

a = torch.rand(D, dtype=torch.float64) + 0.1
b = torch.rand(D, dtype=torch.float64) + 0.1
d = frechet_distance(torch.zeros(D), torch.diag(a), torch.zeros(D), torch.diag(b))
assert abs(d - ((a.sqrt() - b.sqrt()) ** 2).sum().item()) < 1e-8, "diagonal case: sum (sqrt(a) - sqrt(b))^2"

mu_f, s_f = feature_statistics(fake_features(5000, seed=2, shift=0.3, scale=1.2))
assert abs(frechet_distance(mu, sigma, mu_f, s_f) - frechet_distance(mu_f, s_f, mu, sigma)) < 1e-8, "symmetric"
print("Test 2 passed: zero, translation, diagonal closed form, symmetry")

In [ ]:
# Test 3: random SPD matrices vs an independent reference, and the rank-deficient case
def reference_fid(mu1, s1, mu2, s2):
    """Different route: complex eigenvalues of the non-symmetric product, principal sqrt, real part."""
    mu1, s1, mu2, s2 = (t.double() for t in (mu1, s1, mu2, s2))
    tr = torch.linalg.eigvals(s1 @ s2).sqrt().real.sum()
    return ((mu1 - mu2) @ (mu1 - mu2) + s1.trace() + s2.trace() - 2 * tr).item()


g = torch.Generator().manual_seed(3)
for dim in (8, 32):
    A1, A2 = torch.randn(dim, 2 * dim, generator=g), torch.randn(dim, 2 * dim, generator=g)
    s1, s2 = A1 @ A1.T / dim, 3 * A2 @ A2.T / dim
    m1, m2 = torch.randn(dim, generator=g), torch.randn(dim, generator=g)
    got, want = frechet_distance(m1, s1, m2, s2), reference_fid(m1, s1, m2, s2)
    assert abs(got - want) < 1e-6 * max(1.0, want), (dim, got, want)

f1, f2 = real_features(20, seed=4), fake_features(20, seed=5, shift=0.5)   # N = 20 < D = 64
d = frechet_distance(*feature_statistics(f1), *feature_statistics(f2))
assert math.isfinite(d) and d > 0, f"rank-deficient covariances must still give a finite FID, got {d}"
print("Test 3 passed: matches the eigvals reference; finite for singular covariances")

In [ ]:
# FID reacts to both mean shifts and covariance mismatch
print("shift  scale   FID")
for shift, scale in ((0.0, 1.0), (0.1, 1.0), (0.3, 1.0), (0.0, 1.2), (0.0, 0.8), (0.3, 1.2)):
    fake = fake_features(5000, seed=6, shift=shift, scale=scale)
    print(f"{shift:5.1f}  {scale:5.1f}  {frechet_distance(mu, sigma, *feature_statistics(fake)):7.3f}")

In [ ]:
# FID is biased upward for small N: two samples from the SAME distribution still score > 0,
# and the bias shrinks roughly like 1/N (here D = 64; Inception's D = 2048 makes it much worse).
print("    N   FID(real sample A, real sample B)")
for n in (100, 400, 1600, 6400):
    d = frechet_distance(*feature_statistics(real_features(n, seed=10)),
                         *feature_statistics(real_features(n, seed=11)))
    print(f"{n:5d}   {d:.4f}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("fid", feature_statistics=feature_statistics, frechet_distance=frechet_distance)

## Why it works

**The trace trick.** Let $S = \Sigma_1^{1/2}$ (symmetric PSD). If $\Sigma_1$ is invertible, $\Sigma_1\Sigma_2 = S\,(S\,\Sigma_2\,S)\,S^{-1}$, so $\Sigma_1\Sigma_2$ is *similar* to $M = S\Sigma_2 S$ and has the same eigenvalues. $M$ is symmetric PSD, so its eigenvalues $\lambda_i \ge 0$ are real, the principal square root $(\Sigma_1\Sigma_2)^{1/2}$ has eigenvalues $\sqrt{\lambda_i}$, and $\operatorname{Tr}(\Sigma_1\Sigma_2)^{1/2} = \sum_i \sqrt{\lambda_i}$. If $\Sigma_1$ is singular the similarity argument fails, but $\Sigma_1\Sigma_2 = S\,(S\Sigma_2)$ and $(S\Sigma_2)\,S = M$ share their non-zero eigenvalues ($AB$ and $BA$ always do), and the zero ones contribute nothing to the trace — so the formula still holds. Equivalently, $\operatorname{Tr}(\Sigma_1\Sigma_2)^{1/2} = \lVert \Sigma_1^{1/2}\Sigma_2^{1/2}\rVert_*$, the nuclear norm (sum of singular values).

**Why `scipy.linalg.sqrtm` returns complex numbers.** `sqrtm` works on the general, non-symmetric matrix $\Sigma_1\Sigma_2$ through a (complex) Schur decomposition. In exact arithmetic its eigenvalues are real and $\ge 0$, but with finite precision — especially when the covariances are near-singular ($N$ comparable to or smaller than 2048) — some come out slightly negative or as complex-conjugate pairs, and their square roots are complex. `pytorch-fid` therefore retries with $\varepsilon I$ added to both covariances if `sqrtm` returns non-finite values, and drops a small imaginary part of the result (raising if it is large). Going through the symmetric $M$ avoids that: `eigh` / `eigvalsh` return real eigenvalues by construction and we clamp the tiny negative ones.

**Why float64.** FID is a small difference of large traces ($\operatorname{Tr}\Sigma_1 + \operatorname{Tr}\Sigma_2$ vs. $2\sum\sqrt{\lambda_i}$). With float32's ~7 significant digits the cancellation can wipe out the answer when traces are $10^4$–$10^5$ times larger than the FID.

## Complexity

`feature_statistics`: $O(N D^2)$. `frechet_distance`: two symmetric eigendecompositions and two $D\times D$ matmuls, $O(D^3)$ — for Inception's $D = 2048$ a few seconds at most on a CPU, negligible next to running the network on 50k images.

## Common bugs

- **Biased covariance** (divide by $N$): small but systematic difference from every reference implementation.
- **`eigvalsh(sigma1 @ sigma2)`**: the product is not symmetric, the routine silently reads one triangle and returns the wrong spectrum (and an asymmetric "distance").
- **Elementwise `sqrt`** of $\Sigma_1\Sigma_2$ (or of its diagonal): correct only for diagonal covariances.
- **No clamping** before `sqrt`: a $-10^{-15}$ eigenvalue becomes NaN, which then poisons the next decomposition.
- **float32 throughout**: the trace cancellation loses most significant digits.
- **`||mu1 - mu2||` instead of its square**, or **missing the factor 2** on the cross term.

## Interview follow-ups

1. **What does FID assume, and why 50k samples?** It models each feature distribution as a single Gaussian (only first and second moments), so it is blind to differences in higher moments and to mode structure inside a Gaussian blob. The estimator is biased upward with a bias $\approx c/N$ that depends on the generator (Chong & Forsyth, 2020), so FIDs computed with different $N$ are not comparable — hence the 50k-sample convention (and extrapolating to $N\to\infty$, "FID∞", as a fix).
2. **FID vs IS vs CLIP score vs precision/recall?** IS only looks at generated images (confident and diverse class predictions) and ignores the real distribution; FID compares to real data but mixes fidelity and diversity into one number; precision/recall (Kynkäänniemi et al.) and density/coverage separate the two using k-NN manifolds in feature space; CLIP score measures text–image alignment for conditional models; KID (an unbiased MMD with a polynomial kernel) is preferred for small $N$.
3. **Why do two papers get different FIDs for the same model?** The number depends on the feature extractor (TF Inception weights vs torchvision), image resizing (bilinear vs bicubic, antialiasing), JPEG compression, the reference set and its preprocessing, and $N$. clean-fid (Parmar et al.) standardized the resizing; always report which implementation you used.
4. **What does FID miss?** Memorization (copying training images scores very well), small-scale artifacts that Inception features ignore, and ImageNet-class bias — Inception features over-weight object categories, so it can be gamed by matching class histograms (Kynkäänniemi et al., 2023). Newer work uses DINOv2 features (FD_DINOv2) for this reason.
5. **How would you compute FID for 1M images efficiently?** Stream features and accumulate $\sum x$ and $\sum x x^\top$ in float64 (or use a Welford / batched covariance update) instead of storing all features; the final distance is a single $O(D^3)$ step.